# Import Libraries

In [114]:
import pandas as pd
import numpy as np
import spacy
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Input, Dropout, GlobalMaxPooling1D, Conv1D, Bidirectional, BatchNormalization, SimpleRNN, Attention, GlobalAveragePooling1D, Bidirectional, MaxPool1D
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.metrics import F1Score
from tensorflow.keras.optimizers import Adam
from gensim.models import Word2Vec
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [2]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

In [3]:
# Tokenize input text
# Load BERT tokenizer and model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = TFBertModel.from_pretrained(model_name)

def tokenize_texts(texts, max_len):
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='tf'
    )

    outputs = bert_model(encodings)

    return outputs.last_hidden_state

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [4]:
def make_dataset(texts, labels, tokenizer, shuffle=False, max_len= 20, batch_size = 32):
    def gen():
        for t, l in zip(texts, labels):
            enc = tokenizer(
                t,
                truncation=True,
                padding='max_length',
                max_length=max_len,
                return_tensors='tf'
            )
            yield ({'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0]}, l)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'input_ids': tf.TensorSpec(shape=(max_len,), dtype=tf.int32),
             'attention_mask': tf.TensorSpec(shape=(max_len,), dtype=tf.int32)},
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Pre-Processing

### Import Data

In [233]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
df = pd.DataFrame()
for i in [1,2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

max_len = max(len(tokenizer.encode(text, add_special_tokens=True)) for text in df['question'])
print("Max sequence length:", max_len)

Max sequence length: 95


In [213]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

test_df['processed_question'] = test_df['question'].apply(preprocess_text)
test_df['processed_question'] = [''.join(text) for text in test_df['processed_question']]

## Tokenize

### BERT

In [234]:
# Parameters
num_classes = 6

# Embedding
x_train = tokenize_texts(df['question'], max_len)
x_test = tokenize_texts(test_df['question'], max_len)

In [235]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = df['label'].map(y_mapper)
y_test_mapped = test_df['label'].map(y_mapper)

y_train = to_categorical(np.asarray(y_mapped))
y_test = to_categorical(np.asarray(y_test_mapped))

# Modelling

In [236]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
]


## 1D CNN

In [237]:
cnn_model = Sequential([
    # Input(shape=(max_len, 768)),

    Conv1D(128, 5, activation='gelu', padding= 'same', input_shape=(max_len, 768)),
    BatchNormalization(),
    Conv1D(64, 3, activation='gelu'),
    BatchNormalization(),

    GlobalMaxPooling1D(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='sigmoid'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

cnn_model.summary()

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_63"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_35 (Conv1D)              │ (None, 95, 128)        │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_35          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_36 (Conv1D)              │ (None, 93, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 93, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_22         │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_209 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_174 (Dense)               │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_210 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_175 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_211 (Dropout)           │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_176 (Dense)               │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 523,494 (2.00 MB)

 Trainable params: 523,110 (2.00 MB)

 Non-trainable params: 384 (1.50 KB)

In [238]:
cnn_history = cnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split= 0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.1340 - f1_macro: 0.0984 - loss: 2.2535 - val_accuracy: 0.0891 - val_f1_macro: 0.0273 - val_loss: 1.9142 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1995 - f1_macro: 0.1655 - loss: 1.9768 - val_accuracy: 0.1941 - val_f1_macro: 0.1154 - val_loss: 1.6717 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.2358 - f1_macro: 0.1868 - loss: 1.8896 - val_accuracy: 0.6139 - val_f1_macro: 0.2505 - val_loss: 1.5269 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.2437 - f1_macro: 0.1963 - loss: 1.8067 - val_accuracy: 0.6257 - val_f1_macro: 0.2251 - val_loss: 1.4384 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.2688 - f1_macro: 0.2253 - loss: 1.7912 - val_accuracy: 0.6218 - val_f1_macro: 0.2162 - val_loss: 1.3672 - learning_rate: 1.0000e-04
Epoch 6/20

In [250]:
history_dict = cnn_history.history

val_acc = history_dict['val_accuracy'][-20:]
val_f1 = history_dict['val_f1_macro'][-20:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7624, min=0.7089, avg=0.7355
Val F1:       max=0.6780, min=0.6367, avg=0.6627


In [220]:
results = cnn_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")


Test Loss: 1.5173
Test Accuracy: 42.17%
Test F1 Macro: 0.4092


## RNN

In [240]:
rnn_model = Sequential([
        Input(shape=(max_len, 768)),
        
        SimpleRNN(128, activation= 'tanh', return_sequences=True),
        Dropout(0.3),
        
        SimpleRNN(64, activation= 'tanh'),
        Dropout(0.3),
        
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='sigmoid'),
        Dropout(0.4),
        
        Dense(6, activation='softmax')
    ])

rnn_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

rnn_model.summary()

Model: "sequential_64"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_38 (SimpleRNN)       │ (None, 95, 128)        │       114,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_212 (Dropout)           │ (None, 95, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_39 (SimpleRNN)       │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_213 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_177 (Dense)               │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_214 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_178 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_215 (Dropout)           │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_179 (Dense)               │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,606 (521.90 KB)

 Trainable params: 133,606 (521.90 KB)

 Non-trainable params: 0 (0.00 B)

In [241]:
rnn_history = rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.1891 - f1_macro: 0.1270 - loss: 1.9383 - val_accuracy: 0.6000 - val_f1_macro: 0.1323 - val_loss: 1.5416 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2240 - f1_macro: 0.1436 - loss: 1.8643 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.4963 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.2554 - f1_macro: 0.1645 - loss: 1.8446 - val_accuracy: 0.6020 - val_f1_macro: 0.1327 - val_loss: 1.4598 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.2816 - f1_macro: 0.1744 - loss: 1.8095 - val_accuracy: 0.6040 - val_f1_macro: 0.1516 - val_loss: 1.4504 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.2792 - f1_macro: 0.1913 - loss: 1.7999 - val_accuracy: 0.6040 - val_f1_macro: 0.1555 - val_loss: 1.4366 - learning_rate: 1.0000e-04
Epoch 6/20

In [251]:
history_dict = rnn_history.history

val_acc = history_dict['val_accuracy'][-20:]
val_f1 = history_dict['val_f1_macro'][-20:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")

Val Accuracy: max=0.6198, min=0.5010, avg=0.5907
Val F1:       max=0.4139, min=0.3510, avg=0.3928


In [224]:
results = rnn_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.6771
Test Accuracy: 31.67%
Test F1 Macro: 0.2395


## LSTM

In [243]:
lstm_model = Sequential([
    Input(shape=(max_len, 768)),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),

    Bidirectional(LSTM(64, return_sequences=True)), 
    GlobalAveragePooling1D(),                         
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])


lstm_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

lstm_model.summary()

Model: "sequential_65"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_44                │ (None, 95, 256)        │       918,528 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_216 (Dropout)           │ (None, 95, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_45                │ (None, 95, 128)        │       164,352 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_18     │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_217 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_180 (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_218 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_181 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_219 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_182 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,108,038 (4.23 MB)

 Trainable params: 1,108,038 (4.23 MB)

 Non-trainable params: 0 (0.00 B)

In [244]:
lstm_history = lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 9s 114ms/step - accuracy: 0.2317 - f1_macro: 0.1579 - loss: 1.7775 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.4559 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 9s 140ms/step - accuracy: 0.3583 - f1_macro: 0.1928 - loss: 1.6504 - val_accuracy: 0.6416 - val_f1_macro: 0.2638 - val_loss: 1.3021 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 11s 166ms/step - accuracy: 0.4108 - f1_macro: 0.3065 - loss: 1.5184 - val_accuracy: 0.6317 - val_f1_macro: 0.2953 - val_loss: 1.1578 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 10s 160ms/step - accuracy: 0.5230 - f1_macro: 0.4358 - loss: 1.3609 - val_accuracy: 0.6079 - val_f1_macro: 0.4248 - val_loss: 1.1322 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.5391 - f1_macro: 0.4622 - loss: 1.2524 - val_accuracy: 0.6317 - val_f1_macro: 0.5099 - val_loss: 1.0902 - learning_rate: 1.0000e-04
Ep

In [245]:
history_dict = lstm_history.history

val_acc = history_dict['val_accuracy']
val_f1 = history_dict['val_f1_macro']

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7109, min=0.6000, avg=0.6637
Val F1:       max=0.5937, min=0.1250, avg=0.5077


In [228]:
results = lstm_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.6803
Test Accuracy: 41.33%
Test F1 Macro: 0.4100


### Attention + LSTM

In [246]:
inputs = Input(shape=(max_len, 768), name='inputs')
# return_sequences=True so Attention can attend over time
lstm_out = LSTM(64, activation='tanh', return_sequences=True, name='lstm')(inputs)

# self-attention: query = key = value = lstm_out
attn_out = Attention(use_scale=True, name='self_attention')([lstm_out, lstm_out])

# pool to single vector
context = GlobalAveragePooling1D(name='gap')(attn_out)

h = Dense(32, activation='relu', name='dense_1')(context)
h = Dropout(0.2, name='dropout_1')(h)
outputs = Dense(6, activation='softmax', name='output')(h)

at_lstm_model = Model(inputs=inputs, outputs=outputs, name='simple_attn_lstm')
at_lstm_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )
at_lstm_model.summary()

Model: "simple_attn_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, 95, 768)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 95, 64)    │    213,248 │ inputs[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 95, 64)    │          1 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 64)        │          0 │ self_attention[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 6)         │        198 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 215,527 (841.90 KB)

 Trainable params: 215,527 (841.90 KB)

 Non-trainable params: 0 (0.00 B)

In [247]:
at_lstm_history = at_lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.2296 - f1_macro: 0.1199 - loss: 1.7684 - val_accuracy: 0.5980 - val_f1_macro: 0.1322 - val_loss: 1.4151 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.3583 - f1_macro: 0.1706 - loss: 1.6134 - val_accuracy: 0.6277 - val_f1_macro: 0.2410 - val_loss: 1.2762 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.4522 - f1_macro: 0.3226 - loss: 1.4569 - val_accuracy: 0.6752 - val_f1_macro: 0.3753 - val_loss: 1.1779 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.4895 - f1_macro: 0.3828 - loss: 1.3841 - val_accuracy: 0.6594 - val_f1_macro: 0.4237 - val_loss: 1.1045 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.5302 - f1_macro: 0.4538 - loss: 1.2999 - val_accuracy: 0.6554 - val_f1_macro: 0.4649 - val_loss: 1.0629 - learning_rate: 1.0000e-04
Epoch 6/20

In [248]:
history_dict = at_lstm_history.history

val_acc = history_dict['val_accuracy']
val_f1 = history_dict['val_f1_macro']

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7010, min=0.5980, avg=0.6683
Val F1:       max=0.6050, min=0.1322, avg=0.5116


In [232]:
results = at_lstm_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.6739
Test Accuracy: 42.33%
Test F1 Macro: 0.4215
